# NorthWind PLUS business agent — Relate by Neo4j + Neo4j Aura + Claude

Ask business questions about the **live** NorthWind PLUS simulation. The agent looks up proven Cypher templates in its
**relate** knowledge store (the ontology plus the query library), runs read queries on **Neo4j Aura**, and can make one
governed change: the L1/L2/L3 PO approval bands.

**Before you start**
1. Load the graph: in the Aura Query console, run `model/NorthWind_PLUS_Graph_Data_Model_Import_v2.3.2.cypher`
   from line 95 (*"// Let's clear all Nodes..."*) to line 1171. This **wipes the database** and builds NorthWind PLUS v2.3.2.
   Optionally also run the Recommendation Engine (lines 1271-1334).
   *(Already on v2.3.1 with data you want to keep? Run `model/NorthWind_PLUS_v2.3.2_Upgrade_from_v2.3.1.cypher` instead.)*
2. Run `model/NorthwindPlusGraphType_v2.3.2.cypher` in the same console, to apply the GRAPH TYPE schema.
3. Start the six Python loops (`model/NorthwindPlus_Stress_Test.py`, one process per `--loop`), so the data is moving.
4. Fill in `.env` in this folder (see `.env.example`): `ANTHROPIC_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`, `NEO4J_DATABASE`.
5. Use a kernel where `pip install -r requirements.txt` has been run.

Run the cells from the top. Each question is one `ask(...)` call.

## 1. Setup

In [ ]:
import logging, os, sys, json
from pathlib import Path

ROOT = Path.cwd()                      # the notebook lives in the project folder
assert (ROOT / "agent" / "agent.py").exists(), f"Open this notebook from the project folder (found {ROOT})"
for sub in ("agent", "kb"):
    sys.path.insert(0, str(ROOT / sub))

logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)   # quiet "property does not exist" warnings

from IPython.display import Markdown, display
import agent as northwind_agent
from build_kb import build
print("Project folder:", ROOT)

## 2. Knowledge store (relate)

Builds `kb/northwind_kb.rl` from the model files if it doesn't exist yet. Set `REBUILD = True` after you edit
`kb/semantics.json` or the model files.

In [ ]:
REBUILD = False

store = ROOT / "kb" / "northwind_kb.rl"
if REBUILD or not store.exists():
    print(json.dumps(build(store), indent=2))
else:
    print(f"Using existing knowledge store: {store.name}")

## 3. Sign in

`EMPLOYEE_ID` stands in for a login — the session decides who you are, the model never does.

| EmployeeID | Who | Can change approval bands? |
|---|---|---|
| `25` | Olivia Bennett, ProcurementPolicyAdmin | yes |
| `12` | Gloria Gaynor, Level1Approver | no (can still ask questions) |

In [ ]:
EMPLOYEE_ID = "25"
MODEL = northwind_agent.DEFAULT_MODEL      # e.g. "claude-opus-5-5" for harder questions

session = northwind_agent.Session(EMPLOYEE_ID, MODEL)
print(f"Signed in as {session.user['name']} ({', '.join(session.user['roles']) or 'no roles'}) -- model {session.model}")
print(f"Session log: {session.log_path.relative_to(ROOT)}")


def ask(question: str):
    """Ask the agent. Tool calls print as they happen; the answer renders as Markdown."""
    answer = session.ask(question)
    display(Markdown(answer))


def new_conversation():
    """Forget the conversation so far (same sign-in, same log file)."""
    session.messages.clear()
    print("Conversation cleared.")

## 4. Ask questions

The conversation carries over between cells, so follow-ups like *"and for last week?"* work.

In [ ]:
ask("Which suppliers get their quotes rejected most often?")

In [ ]:
ask("How many POs are waiting for approval?")

### Governed change

The agent previews first, then asks you to confirm. In Jupyter the confirmation appears as a text box under the cell:
type `y` and press Enter to apply, anything else to cancel. Only EmployeeID 25 can apply it.

In [ ]:
ask("Raise L1 to 8000 and L2 to 20000 to speed up vetting.")

In [ ]:
ask("Who changed the approval limits, when and why?")

### Your questions

In [ ]:
# ask("Which products are at risk of running out?")

## 5. What happened in this session

Every question, tool call and answer is in the session log. This summary is handy for the article: which templates
were reused, which queries were new, and what each answer cost in tokens.

In [ ]:
events = [json.loads(line) for line in session.log_path.read_text(encoding="utf-8").splitlines()]
rows, totals = [], {"input_tokens": 0, "output_tokens": 0, "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0}
question = None
for e in events:
    if e["event"] == "question":
        question, tools = e["text"], []
    elif e["event"] == "tool":
        label = e["name"]
        if e["name"] == "run_read_query":
            label += f" ({e['input'].get('template') or 'new Cypher'})"
        tools.append(f"{label} {e['ms']} ms")
    elif e["event"] == "answer":
        u = e.get("usage") or {}
        for k in totals:
            totals[k] += u.get(k) or 0
        rows.append(f"| {question} | {'<br>'.join(tools) or '-'} | {u.get('model_calls', 1)} calls: {u.get('input_tokens', 0):,} in (+{u.get('cache_read_input_tokens') or 0:,} cached) / {u.get('output_tokens', 0):,} out |")
display(Markdown("| Question | Tool calls | Tokens (all model calls) |\n|---|---|---|\n" + "\n".join(rows)))
print("Session totals:", totals)

## 6. Done

In [ ]:
session.aura.close()
session.kb.close()
print("Closed the Aura connection and the knowledge store.")